In [4]:
import os
import numpy as np
import pandas as pd
import scipy.io as sio
from matplotlib import pyplot as plt

In [6]:
data_dir = 'training2017'
target_len = 9000

ref = pd.read_csv(
    os.path.join(data_dir, 'REFERENCE-original.csv'),
    header=None,
    names=['record', 'label']
)

def fix_length(v, target_len):
    v = np.squeeze(v)
    n = v.size

    if n == target_len:
        return v

    if n > target_len:
        # crop centrale (puoi scegliere anche dall'inizio: v[:target_len])
        start = (n - target_len) // 2
        return v[start:start + target_len]

    # n < target_len: padding con zeri in coda
    pad_width = target_len - n
    return np.pad(v, (0, pad_width))

signals = []
labels = []

for rec, lab in zip(ref['record'], ref['label']):
    mat_path = os.path.join(data_dir, rec + '.mat')
    if not os.path.exists(mat_path):
        continue

    d = sio.loadmat(mat_path)
    v = d['val']   # (1, n) in generale

    v_fixed = fix_length(v, target_len)
    signals.append(v_fixed)
    labels.append(lab)

X = np.stack(signals, axis=0)   # (N_record, 9000)
y = np.array(labels)

print(X.shape, y.shape)

(8528, 9000) (8528,)


In [ ]:
i = 0  # indice del record che vuoi vedere
v = X[i]
fs = 300
t = np.arange(v.size) / fs

plt.figure(figsize=(10, 4))
plt.plot(t, v)
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')
plt.title(f'ECG sample #{i} - label {y[i]}')
plt.grid(True)
plt.show()